In [ ]:
# command to auto-reload modules when they are edited (easier for testing and debugging)
%load_ext autoreload
%autoreload 2

In [ ]:
import healpix_geo 
import numpy as np
from foscat.Plot import lgnomproject
import matplotlib.pyplot as plt

level=15
ilevel=15
ndata=128
lon,lat=np.meshgrid(0.3*np.arange(ndata)/ndata,0.3*np.arange(ndata)/ndata)
#cell_ids=3*4**ilevel+np.arange(ndata*ndata)
#cell_ids=np.max(cell_ids)+1
#lon,lat = healpix_geo.nested.healpix_to_lonlat(cell_ids, ilevel,ellipsoid='WGS84')
lon=lon.T.ravel()
lat=lat.T.ravel()
nline=32
lon=np.concatenate([-0.2*np.arange(nline)/nline,lon])
lat=np.concatenate([0.0*np.arange(nline)/nline,lat])
val=lon

In [ ]:
from regrid_to_healpix.zuniq import Set

nr = Set(lon_deg=lon, lat_deg=lat)  # zuniq transform
# puis:
hval     = nr.transform(val)  # selon ton API
rval     = nr.invert(hval)
cell_ids = nr.get_cell_ids()
np.max(rval-val)

In [ ]:
import healpix_geo
lo,la=healpix_geo.nested.healpix_to_lonlat(cell_ids, 29,ellipsoid='WGS84')

In [ ]:
lo[lo>180]-=360
plt.scatter(lon,lat)
plt.scatter(lo,la,color='r',s=1)

In [ ]:
plt.figure(figsize=(12,6))
plt.scatter(lon,lon,color='b',s=4)
plt.scatter(lon,rval,color='r',s=1)
np.mean((rval-lon)**2)

In [ ]:
from regrid_to_healpix.bilinear import Set

nr = Set(lon_deg=lon, lat_deg=lat, level=level)  # Npt=1 imposé dans nearest
# puis:
hval     = nr.transform(val,lam=0.0)  # selon ton API
rval     = nr.invert(hval)
cell_ids = nr.get_cell_ids()
hval.shape,rval.shape

In [ ]:
plt.figure(figsize=(12,6))
lgnomproject(cell_ids,hval,2**(level),hold=False,fov_deg=1,interp=False,sub=(1,2,1),cbar=True)
plt.subplot(1,2,2)
plt.scatter(lon,lon,color='b',s=4)
plt.scatter(lon,rval,color='r',s=1)
np.mean((rval-lon)**2)
cell_ids_from_bili = cell_ids

In [ ]:
from regrid_to_healpix.psf import Set

nr = Set(lon_deg=lon, lat_deg=lat, level=level,threshold=0.5,verbose=True)  # Npt=1 imposé dans nearest
# puis:
hval     = nr.transform(val,lam=0.0)  # selon ton API
rval     = nr.invert(hval)
cell_ids = nr.get_cell_ids()
hval.shape,rval.shape

In [ ]:
plt.figure(figsize=(12,6))
lgnomproject(cell_ids,hval,2**(level),hold=False,fov_deg=1,interp=False,sub=(1,2,1),cbar=True)
plt.subplot(1,2,2)
plt.scatter(lon,lon,color='b',s=4)
plt.scatter(lon,rval,color='r',s=1)
np.mean((rval-lon)**2)

In [ ]:
from regrid_to_healpix.nearest import Set

nr = Set(lon_deg=lon, lat_deg=lat, level=level)  # Npt=1 imposé dans nearest
# puis:
hval     = nr.transform(val)  # selon ton API
rval     = nr.invert(hval)
cell_ids = nr.get_cell_ids()
hval.shape,rval.shape

In [ ]:

plt.figure(figsize=(12,6))
lgnomproject(cell_ids,hval,2**(level),hold=False,fov_deg=1,interp=False,sub=(1,2,1),cbar=True)
plt.subplot(1,2,2)
plt.plot(lon,color='b',lw=4)
plt.plot(rval,color='r')
np.mean((rval-lon)**2)

In [ ]:
ilevel=14
ndata=128
cell_ids=3*4**ilevel+np.arange(ndata*ndata)
val=np.cos(cell_ids/100.) # fake data
lgnomproject(cell_ids,val,2**(ilevel),hold=False,fov_deg=1,interp=False,sub=(1,2,1),cbar=True)

In [ ]:
import xarray as xr
zarr_file_path = 'https://data-fair2adapt.ifremer.fr/riomar-zarr_tina/small.zarr'
small_ds=xr.open_zarr(zarr_file_path)
small_ds
ds=small_ds.temp.isel(s_rho=0,time_counter=0).compute()
ds = ds.stack(point=("y_rho","x_rho")).dropna(dim="point")
ds

In [ ]:
import regrid_to_healpix.bilinear as Bili
import regrid_to_healpix.psf as Psf

child_level=13

# Build operator once
lon = ds["nav_lon_rho"].values.astype(np.float64)
lat = ds["nav_lat_rho"].values.astype(np.float64)

nr_bili = Bili.Set(lon_deg=lon, lat_deg=lat, level=child_level, device="cuda", threshold=0.5, ellipsoid="WGS84")
nr_psf = Psf.Set(lon_deg=lon, lat_deg=lat, level=child_level, device="cuda", threshold=0.5, ellipsoid="WGS84",out_cell_ids=nr_bili.get_cell_ids(),verbose=True)

cell_ids=nr_psf.get_cell_ids()
ncell = int(cell_ids.size)

data1d = np.asarray(ds.compute(), dtype=np.float64)

def to_healpix_point(data_1d,nr):
    out = nr.transform(data_1d, lam=0.1)#.1)
    return np.asarray(out, dtype=np.float64)
    
ar1=to_healpix_point(data1d,nr_psf)
ar2=to_healpix_point(data1d,nr_bili)

In [ ]:
#v=nr_psf.M.to_dense().cpu().numpy()
import torch

r=(torch.tensor(data1d,device='cuda',dtype=torch.float)@nr_psf.M).cpu().numpy()
rinv=(torch.tensor(r,device='cuda',dtype=torch.float)@nr_psf.MT).cpu().numpy()
plt.figure(figsize=(16,6))
lgnomproject(cell_ids,r,2**(child_level),fov_deg=0.8,interp=False,cbar=True,hold=False,sub=(1,3,1),cmap='jet',title='y@M')
plt.subplot(1,3,2)
plt.title('(y@M)@MT')
plt.scatter(lon,lat,c=rinv,s=10,cmap='jet')
plt.colorbar(location='bottom')
plt.subplot(1,3,3)
plt.title('y-(y@M)@MT')
plt.scatter(lon,lat,c=rinv-data1d,s=10,cmap='jet')
plt.colorbar(location='bottom')

In [ ]:
plt.figure(figsize=(16,7))
lgnomproject(cell_ids,(ar2),2**(child_level),hold=False,sub=(1,3,1),fov_deg=0.8,interp=False,cbar=True,cmap='jet',title='Bilinear')
lgnomproject(cell_ids,(ar1),2**(child_level),hold=False,sub=(1,3,2),fov_deg=0.8,interp=False,cbar=True,cmap='jet',title='PSF')
lgnomproject(cell_ids,(ar1-ar2),2**(child_level),hold=False,sub=(1,3,3),fov_deg=0.8,interp=False,cbar=True,cmap='jet',title='Bilinear-PSF')

In [ ]:
rval1     = nr_psf.invert(ar1)
rval2     = nr_bili.invert(ar2)

In [ ]:
plt.figure(figsize=(16,6))
plt.subplot(1,2,2)
plt.title('Residual PSF')
plt.scatter(lon,lat,c=np.log10(abs(ds.compute()-rval1)+1e-7),s=10,vmin=-7,vmax=1,cmap='jet')
plt.colorbar(label=r'$log10(\Delta)$')
plt.subplot(1,2,1)
plt.title('Residual Bilinear')
plt.scatter(lon,lat,c=np.log10(abs(ds.compute()-rval2)),s=10,vmin=-7,vmax=1,cmap='jet')
plt.colorbar(label=r'$log10(\Delta)$')